# Genomics, Sequence Analysis, and Biological Data Workflow

This notebook scaffold mirrors the repository workflow: FASTA parsing, GC content, k-mer counting, FASTQ quality summaries, variant validation, metadata checks, and provenance documentation.

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd

article_dir = Path.cwd().parent
fasta_text = (article_dir / 'data' / 'sequences.fasta').read_text()

records = {}
current = None
sequence = []
for line in fasta_text.strip().splitlines():
    if line.startswith('>'):
        if current is not None:
            records[current] = ''.join(sequence).upper()
        current = line[1:].split()[0]
        sequence = []
    else:
        sequence.append(line.strip())
if current is not None:
    records[current] = ''.join(sequence).upper()

records

In [ ]:
valid = set('ACGT')
def gc_content(sequence):
    bases = [base for base in sequence if base in valid]
    counts = Counter(bases)
    return (counts['G'] + counts['C']) / len(bases)

summary = pd.DataFrame({
    'sequence_id': sequence_id,
    'length': len(sequence),
    'gc_content': gc_content(sequence),
    'ambiguous_bases': sum(base not in valid for base in sequence),
} for sequence_id, sequence in records.items())
summary.round(5)

In [ ]:
variants = pd.read_csv(article_dir / 'data' / 'variants.csv')
variants['variant_allele_frequency'] = variants['alternate_depth'] / variants['read_depth']
variants['passes_depth_threshold'] = variants['read_depth'] >= 10
variants.round(5)